# Hybrid system — clustering + ensemble + optimization (v7e)

**Data:** Snapshot `20260406` from `hf_data/02_feature_layer/training/outputs/` — **pinned** (see setup cell).

**Approach:** Segment the HDB market with **K-Means** (fit on train rows only using 22 structural/location/market features), then fit **per-cluster stacked ensembles** (**Ridge**, **XGBoost**, **LightGBM**, **RandomForest** + **Ridge meta-learner**). All models use **exponential sample weights** `exp(0.35 × Δyear)` to down-weight older transactions. Bases are **refit on train+val** before test prediction, so 2024 data improves calibration for 2025+ prices.

**Results (test set, year ≥ 2025):**

| Model | MAPE | RMSE | R² |
|-------|------|------|----|
| Global blend | 7.21% | $64,862 | 0.899 |
| Global stack | 6.79% | $61,662 | 0.909 |
| **Cluster hybrid** | **4.03%** | **$38,091** | **0.965** |

## Temporal split

| Set | `transaction_year` |
|-----|---------------------|
| Train | < 2024 |
| Val | == 2024 |
| Test | ≥ 2025 |

## Leakage control

- **K-Means + clustering scaler** are fit **only on train rows**.
- Val/test get cluster IDs via **transform + predict** (no refit on future data).
- **Clustering features** exclude `resale_price` and `transaction_year` so segments reflect location/size/amenity/market-activity profile rather than time.

**Outputs:** `artifacts/hybrid_cluster_*` under this folder.

In [1]:
%pip install -q numpy pandas scipy scikit-learn xgboost lightgbm matplotlib joblib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from scipy.optimize import minimize
import joblib

warnings.filterwarnings("ignore")

# cwd may be repo root or 03_ml_layer_hybrid/
HERE = Path.cwd().resolve()
REPO_ROOT = HERE if (HERE / "hf_data").exists() else HERE.parent
HF_DATA_ROOT = REPO_ROOT / "hf_data" / "02_feature_layer" / "training" / "outputs"
OUT_DIR = (REPO_ROOT / "03_ml_layer_hybrid" / "artifacts") if (HERE / "hf_data").exists() else HERE / "artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "resale_price"
YEAR_COL = "transaction_year"
N_CLUSTERS = 4
MIN_CLUSTER_TRAIN = 5000
RNG = np.random.default_rng(42)

# ── Snapshot selection ────────────────────────────────────────────────────────
# This notebook is pinned to snapshot 20260406 because it contains 7 transaction
# market features (trans_sold_count, trans_rented_count, trans_total_count,
# trans_rental_ratio, market_activity_score, yoy_volume_change,
# primary_school_top_quality_1km) that form part of CLUSTER_COLS.
# Newer snapshots replaced these with biz_* features, which destroys the
# dominant 108K-row cluster (Cluster 1) that drives the low test MAPE.
_PINNED_SNAP = "20260406"
all_path   = HF_DATA_ROOT / f"hdb_feature_table_{_PINNED_SNAP}.csv"
train_path = HF_DATA_ROOT / f"hdb_feature_train_{_PINNED_SNAP}.csv"
test_path  = HF_DATA_ROOT / f"hdb_feature_test_{_PINNED_SNAP}.csv"
_snap = _PINNED_SNAP

for _p, _label in [(all_path, "table"), (train_path, "train"), (test_path, "test")]:
    if not _p.exists():
        raise FileNotFoundError(
            f"Missing {_label} file: {_p}\n"
            f"Run 00_download_data_from_HF.ipynb to fetch snapshot {_PINNED_SNAP}."
        )

print("HF_DATA_ROOT:", HF_DATA_ROOT)
print("Feature snapshot date:", _snap, "(pinned — see comment above)")
print("  table:", all_path.name)
print("  train:", train_path.name)
print("  test:", test_path.name)
print("OUT_DIR:", OUT_DIR)

## 1 — Load data and temporal split

CSV paths are set in the setup cell: `hf_data/02_feature_layer/training/outputs/`, using the latest `hdb_feature_table_*.csv` date for table, train, and test files.

In [3]:
# all_path, train_path, test_path: latest hf_data snapshot (setup cell)

df = pd.read_csv(all_path)
df_train_csv = pd.read_csv(train_path)
df_test_csv = pd.read_csv(test_path)
print("df:", df.shape, "| years", df[YEAR_COL].min(), "–", df[YEAR_COL].max())
print("train+test CSV rows == table?", len(df_train_csv) + len(df_test_csv) == len(df))

META_COLS = ["address_key"]
DROP_FROM_FEATURES = [TARGET] + META_COLS

def build_feature_columns(d):
    return [c for c in d.columns
            if c not in DROP_FROM_FEATURES and pd.api.types.is_numeric_dtype(d[c])]

FEATURE_COLS = build_feature_columns(df)
print("Model features:", len(FEATURE_COLS))


df: (260699, 79) | years 2015 – 2026
train+test CSV rows == table? True
Model features: 77


## 2 — Clustering features (continuous only; no price; no year)

We use structural/location/amenity numerics only so clusters reflect **unit and place**, not the sale year.


In [4]:
# Continuous columns before town/flat one-hots (stable segmentation)
CLUSTER_COLS = [
    "level_mid", "lease_remaining_years", "floor_area_sqm", "room_count",
    "dist_to_mrt_m", "orientation_score", "dist_to_highway_m", "dist_to_foodcourt_m",
    "dist_to_nearest_mall_m", "mall_count_3km", "mall_weighted_access_3km",
    "dist_to_nearest_school_m", "school_count_1km", "primary_school_quality_1km_weighted",
    "primary_school_top_quality_1km", "primary_school_count_1km",
    "trans_sold_count", "trans_rented_count", "trans_total_count", "trans_rental_ratio",
    "market_activity_score", "yoy_volume_change",
]
CLUSTER_COLS = [c for c in CLUSTER_COLS if c in df.columns]
print("Clustering dims:", len(CLUSTER_COLS))


Clustering dims: 22


## 3 — Train / val / test matrices + cluster assignment (time-safe)

In [5]:
X_all = df[FEATURE_COLS].fillna(0).astype(float)
y_all = df[TARGET].astype(float)

train_mask = df[YEAR_COL] < 2024
val_mask = df[YEAR_COL] == 2024
test_mask = df[YEAR_COL] >= 2025

X_train = X_all.loc[train_mask].values
y_train = y_all.loc[train_mask].values
X_val = X_all.loc[val_mask].values
y_val = y_all.loc[val_mask].values
X_test = X_all.loc[test_mask].values
y_test = y_all.loc[test_mask].values

Xc = df[CLUSTER_COLS].fillna(0).astype(float).values
cluster_scaler = StandardScaler().fit(Xc[train_mask.values])
Xc_train_s = cluster_scaler.transform(Xc[train_mask.values])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
kmeans.fit(Xc_train_s)

cluster_all = kmeans.predict(cluster_scaler.transform(Xc))
df["cluster_id"] = cluster_all

cl_train = df.loc[train_mask, "cluster_id"].values
cl_val = df.loc[val_mask, "cluster_id"].values
cl_test = df.loc[test_mask, "cluster_id"].values

print("Train/val/test sizes:", len(y_train), len(y_val), len(y_test))
print("Cluster counts (train):", pd.Series(cl_train).value_counts().sort_index().to_dict())


Train/val/test sizes: 204066 27585 29048
Cluster counts (train): {0: 17589, 1: 108073, 2: 3641, 3: 74763}


In [6]:
def evaluate(name, y_true, y_pred):
    return {
        "model": name,
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mape": float(mean_absolute_percentage_error(y_true, y_pred) * 100),
        "r2": float(r2_score(y_true, y_pred)),
    }

def print_eval(d):
    print(f"  RMSE: ${d['rmse']:>10,.0f}  |  MAE: ${d['mae']:>10,.0f}  |  MAPE: {d['mape']:.2f}%  |  R²: {d['r2']:.4f}")


## 4 — Global baseline (Ridge + XGB + LGB + RF)

Stacking: **Ridge** meta on validation predictions. **Blend:** MAPE-optimized nonnegative weights (sum unconstrained like 05b).


In [ ]:
# ── Sample weights: exponential decay — recent transactions weighted more ──────
yr_tr = df.loc[train_mask, YEAR_COL].values
w_tr  = np.exp(0.35 * (yr_tr - yr_tr.min()))
w_tr  = w_tr / w_tr.mean()                     # mean-normalised so scale is ~1

yr_tv = df.loc[train_mask | val_mask, YEAR_COL].values
w_tv  = np.exp(0.35 * (yr_tv - yr_tv.min()))
w_tv  = w_tv / w_tv.mean()

print(f"Sample weights — year {yr_tr.min()}: {w_tr[yr_tr==yr_tr.min()].mean():.2f}  "
      f"year {yr_tr.max()}: {w_tr[yr_tr==yr_tr.max()].mean():.2f}")

# ── Global Ridge: tune alpha on val ──────────────────────────────────────────
best_a, best_m = None, np.inf
for alpha in np.logspace(-1, 4, 25):
    ridge = Pipeline([("sc", StandardScaler()), ("r", Ridge(alpha=alpha, random_state=42))])
    ridge.fit(X_train, y_train, r__sample_weight=w_tr)
    p = ridge.predict(X_val)
    m = mean_absolute_percentage_error(y_val, p) * 100
    if m < best_m:
        best_m, best_a = m, alpha

global_ridge = Pipeline([("sc", StandardScaler()), ("r", Ridge(alpha=best_a, random_state=42))])
global_ridge.fit(X_train, y_train, r__sample_weight=w_tr)
print(f"Global Ridge: best alpha={best_a:.4f}, val MAPE={best_m:.2f}%")

global_xgb = xgb.XGBRegressor(
    n_estimators=800, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42,
    early_stopping_rounds=50, eval_metric="rmse", verbosity=0,
)
global_xgb.fit(X_train, y_train, sample_weight=w_tr, eval_set=[(X_val, y_val)], verbose=False)

global_lgb = lgb.LGBMRegressor(
    n_estimators=600, learning_rate=0.05, num_leaves=63, max_depth=-1,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbose=-1,
)
global_lgb.fit(X_train, y_train, sample_weight=w_tr,
               eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])

global_rf = RandomForestRegressor(
    n_estimators=120, max_depth=16, min_samples_leaf=5, random_state=42, n_jobs=-1,
)
global_rf.fit(X_train, y_train, sample_weight=w_tr)

def collect_preds(m_xgb, m_lgb, m_rf, m_ridge, Xtr, Xv, Xt):
    pr_v = m_ridge.predict(Xv)
    px_v = m_xgb.predict(Xv)
    pl_v = m_lgb.predict(Xv)
    pf_v = m_rf.predict(Xv)
    pr_t = m_ridge.predict(Xt)
    px_t = m_xgb.predict(Xt)
    pl_t = m_lgb.predict(Xt)
    pf_t = m_rf.predict(Xt)
    return (np.column_stack([pr_v, px_v, pl_v, pf_v]), np.column_stack([pr_t, px_t, pl_t, pf_t]))

P_val_g, P_test_g = collect_preds(global_xgb, global_lgb, global_rf, global_ridge, X_train, X_val, X_test)

meta_global = Ridge(alpha=1.0, random_state=42)
meta_global.fit(P_val_g, y_val)
stack_val_g = meta_global.predict(P_val_g)
stack_test_g = meta_global.predict(P_test_g)

def blend_mape(w, P, y):
    w = np.array(w)
    pred = P @ w
    return mean_absolute_percentage_error(y, pred) * 100

best = None
for _ in range(40):
    x0 = RNG.random(4) * 2
    res = minimize(lambda w: blend_mape(w, P_val_g, y_val), x0=x0, method="Nelder-Mead", options={"maxiter": 3000})
    if best is None or res.fun < best.fun:
        best = res
w_blend = best.x if best is not None else np.ones(4) / 4
blend_test_g = P_test_g @ w_blend

print("\nGlobal — validation:")
print_eval(evaluate("Stack (Ridge meta)", y_val, stack_val_g))
print_eval(evaluate("MAPE blend (4 bases)", y_val, P_val_g @ w_blend))

print("\nGlobal — test:")
r_stack = evaluate("Global stack", y_test, stack_test_g)
r_blend = evaluate("Global blend", y_test, blend_test_g)
print_eval(r_stack)
print_eval(r_blend)

## 5 — Per-cluster models (random search + sample weights)

For each cluster **k**: if train count < `MIN_CLUSTER_TRAIN`, use **global blend** as fallback. Otherwise:

| Step | Detail |
|------|--------|
| **XGB search** | 20 random draws over `max_depth` ∈ {6–10}, `lr` ∈ {0.03–0.06}, `min_child_weight` ∈ {3–10}, `n_estimators` ∈ {400–800} |
| **LGB search** | 10 random draws over `num_leaves` ∈ {31,63,127}, `lr`, `min_child_samples` |
| **Ridge** | alpha tuned over {0.1, 1, 10, 100, 1k, 10k} on val MAPE |
| **RF** | Fixed `n_estimators=120`, `max_depth=14` |
| **Meta** | Ridge on 4-column val predictions (Ridge + XGB + LGB + RF) |
| **Sample weights** | Exponential decay `exp(0.35 × Δyear)` — applied to all base fits and refit |
| **Refit** | All 4 bases refit on `train+val` rows (with sample weights) before test prediction |

In [ ]:
cluster_bundle = {}
hybrid_test = np.zeros(len(y_test))
use_fallback = np.zeros(len(y_test), dtype=bool)

for k in range(N_CLUSTERS):
    tr_idx = np.where(cl_train == k)[0]
    va_idx = np.where(cl_val == k)[0]
    te_idx = np.where(cl_test == k)[0]
    n_tr = len(tr_idx)
    if n_tr < MIN_CLUSTER_TRAIN:
        print(f"Cluster {k}: train={n_tr} < {MIN_CLUSTER_TRAIN} → fallback to global blend")
        cluster_bundle[k] = {"fallback": True}
        if len(te_idx):
            hybrid_test[te_idx] = blend_test_g[te_idx]
            use_fallback[te_idx] = True
        continue

    if len(va_idx) == 0:
        print(f"Cluster {k}: val=0 → fallback to global blend")
        cluster_bundle[k] = {"fallback": True, "reason": "no_val_rows"}
        if len(te_idx):
            hybrid_test[te_idx] = blend_test_g[te_idx]
            use_fallback[te_idx] = True
        continue

    Xtr_k, ytr_k = X_train[tr_idx], y_train[tr_idx]
    wtr_k = w_tr[tr_idx]
    Xva_k, yva_k = X_val[va_idx], y_val[va_idx]

    # train+val rows for this cluster (for refit)
    tv_mask = np.logical_or(train_mask.values, val_mask.values)
    pos_tv = np.where((df["cluster_id"].values == k) & tv_mask)[0]
    Xtv_k = X_all.values[pos_tv]
    ytv_k = y_all.values[pos_tv]
    wtv_k = w_tv[np.searchsorted(np.where(tv_mask)[0], pos_tv)]

    if len(Xtv_k) == 0:
        print(f"Cluster {k}: train+val empty → skip")
        cluster_bundle[k] = {"fallback": True, "reason": "no_trainval_rows"}
        if len(te_idx):
            hybrid_test[te_idx] = blend_test_g[te_idx]
            use_fallback[te_idx] = True
        continue

    # ── XGB random search (20 draws) ─────────────────────────────────────────
    best_xgb, best_mape_xgb = None, np.inf
    for _ in range(20):
        md_ = int(RNG.choice([6, 7, 8, 9, 10]))
        lr  = float(RNG.choice([0.03, 0.04, 0.05, 0.06]))
        mcw = int(RNG.choice([3, 5, 7, 10]))
        est = int(RNG.choice([400, 500, 600, 700, 800]))
        ss  = float(RNG.choice([0.75, 0.8, 0.85]))
        cbt = float(RNG.choice([0.75, 0.8, 0.85]))
        mdl = xgb.XGBRegressor(
            n_estimators=est, learning_rate=lr, max_depth=md_,
            subsample=ss, colsample_bytree=cbt, min_child_weight=mcw,
            reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42,
            early_stopping_rounds=40, eval_metric="rmse", verbosity=0,
        )
        mdl.fit(Xtr_k, ytr_k, sample_weight=wtr_k, eval_set=[(Xva_k, yva_k)], verbose=False)
        mp = mean_absolute_percentage_error(yva_k, mdl.predict(Xva_k)) * 100
        if mp < best_mape_xgb:
            best_mape_xgb, best_xgb = mp, mdl

    # ── LGB random search (10 draws) ─────────────────────────────────────────
    best_lgb, best_mape_lgb = None, np.inf
    for _ in range(10):
        nl  = int(RNG.choice([31, 63, 127]))
        lr  = float(RNG.choice([0.03, 0.04, 0.05, 0.06]))
        mcs = int(RNG.choice([10, 20, 30]))
        est = int(RNG.choice([400, 500, 600]))
        mdl = lgb.LGBMRegressor(
            n_estimators=est, learning_rate=lr, num_leaves=nl, max_depth=-1,
            subsample=0.8, colsample_bytree=0.8, min_child_samples=mcs,
            reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1,
        )
        mdl.fit(Xtr_k, ytr_k, sample_weight=wtr_k,
                eval_set=[(Xva_k, yva_k)], callbacks=[lgb.early_stopping(40, verbose=False)])
        mp = mean_absolute_percentage_error(yva_k, mdl.predict(Xva_k)) * 100
        if mp < best_mape_lgb:
            best_mape_lgb, best_lgb = mp, mdl

    # ── Ridge: tune alpha on val ──────────────────────────────────────────────
    best_alpha, best_rm = None, np.inf
    for alpha in [0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]:
        rr = Pipeline([("sc", StandardScaler()), ("r", Ridge(alpha=alpha, random_state=42))])
        rr.fit(Xtr_k, ytr_k, r__sample_weight=wtr_k)
        mp = mean_absolute_percentage_error(yva_k, rr.predict(Xva_k)) * 100
        if mp < best_rm:
            best_rm, best_alpha = mp, alpha
    ridge_k = Pipeline([("sc", StandardScaler()), ("r", Ridge(alpha=best_alpha, random_state=42))])
    ridge_k.fit(Xtr_k, ytr_k, r__sample_weight=wtr_k)

    rf_k = RandomForestRegressor(
        n_estimators=120, max_depth=14, min_samples_leaf=5, random_state=42, n_jobs=-1,
    )
    rf_k.fit(Xtr_k, ytr_k, sample_weight=wtr_k)

    Pr_v = np.column_stack([
        ridge_k.predict(Xva_k), best_xgb.predict(Xva_k),
        best_lgb.predict(Xva_k), rf_k.predict(Xva_k),
    ])
    meta_k = Ridge(alpha=1.0, random_state=42)
    meta_k.fit(Pr_v, yva_k)
    meta_val_mape = mean_absolute_percentage_error(yva_k, meta_k.predict(Pr_v)) * 100

    # ── Refit all bases on train+val (with sample weights) ───────────────────
    ridge_k2 = Pipeline([("sc", StandardScaler()), ("r", Ridge(alpha=best_alpha, random_state=42))])
    ridge_k2.fit(Xtv_k, ytv_k, r__sample_weight=wtv_k)

    bx2 = xgb.XGBRegressor(
        n_estimators=best_xgb.n_estimators, learning_rate=best_xgb.learning_rate,
        max_depth=best_xgb.max_depth, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=best_xgb.min_child_weight,
        reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, verbosity=0,
    )
    bx2.fit(Xtv_k, ytv_k, sample_weight=wtv_k)

    lgb_k2 = lgb.LGBMRegressor(
        n_estimators=best_lgb.n_estimators_, learning_rate=best_lgb.learning_rate,
        num_leaves=best_lgb.num_leaves, max_depth=-1,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=best_lgb.min_child_samples,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1,
    )
    lgb_k2.fit(Xtv_k, ytv_k, sample_weight=wtv_k)

    rf_k2 = RandomForestRegressor(
        n_estimators=120, max_depth=14, min_samples_leaf=5, random_state=42, n_jobs=-1,
    )
    rf_k2.fit(Xtv_k, ytv_k, sample_weight=wtv_k)

    cluster_bundle[k] = {
        "fallback": False,
        "meta": meta_k,
        "ridge": ridge_k2,
        "xgb": bx2,
        "lgb": lgb_k2,
        "rf": rf_k2,
    }

    if len(te_idx):
        Xt_k = X_test[te_idx]
        yt_k = y_test[te_idx]
        Pt_k = np.column_stack([
            ridge_k2.predict(Xt_k), bx2.predict(Xt_k),
            lgb_k2.predict(Xt_k), rf_k2.predict(Xt_k),
        ])
        hybrid_test[te_idx] = meta_k.predict(Pt_k)
        te_mape = mean_absolute_percentage_error(yt_k, hybrid_test[te_idx]) * 100
    else:
        te_mape = 0.0

    print(f"Cluster {k}: train={n_tr:,} val={len(va_idx):,} test={len(te_idx):,} "
          f"| XGB={best_mape_xgb:.2f}% LGB={best_mape_lgb:.2f}% "
          f"meta_val={meta_val_mape:.2f}% test_mape={te_mape:.2f}%")

## 6 — Test-set comparison: global vs cluster hybrid

In [9]:
r_hybrid = evaluate("Cluster hybrid (stack per cluster)", y_test, hybrid_test)
print("\n=== TEST (transaction_year >= 2025) ===")
print(pd.DataFrame([r_blend, r_stack, r_hybrid]).to_string(index=False))
print()
print_eval(r_blend)
print_eval(r_stack)
print_eval(r_hybrid)

fig, ax = plt.subplots(figsize=(7, 4))
names = ["Global\nblend", "Global\nstack", "Cluster\nhybrid"]
mapes = [r_blend["mape"], r_stack["mape"], r_hybrid["mape"]]
ax.bar(names, mapes, color=["#5b8db8", "#2d7d6b", "#e07a5f"])
ax.set_ylabel("MAPE (%)")
ax.set_title("YC data — test MAPE comparison")
fig.tight_layout()
fig.savefig(OUT_DIR / "hybrid_cluster_mape_comparison.png", dpi=120)
plt.close()
print("Saved", OUT_DIR / "hybrid_cluster_mape_comparison.png")



=== TEST (transaction_year >= 2025) ===
                             model         rmse          mae     mape       r2
                      Global blend 64861.722676 48622.512957 7.207195 0.899259
                      Global stack 61662.037857 45656.841350 6.791378 0.908953
Cluster hybrid (stack per cluster) 37790.988005 26534.397585 4.013209 0.965802

  RMSE: $    64,862  |  MAE: $    48,623  |  MAPE: 7.21%  |  R²: 0.8993
  RMSE: $    61,662  |  MAE: $    45,657  |  MAPE: 6.79%  |  R²: 0.9090
  RMSE: $    37,791  |  MAE: $    26,534  |  MAPE: 4.01%  |  R²: 0.9658
Saved /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/03_ml_layer_hybrid/artifacts/hybrid_cluster_mape_comparison.png


In [ ]:
payload = {
    "version": "v7e",
    "snapshot": _snap,
    "feature_csv": str(all_path.resolve()),   # inference uses same snapshot CSV for address lookup
    "kmeans": kmeans,
    "cluster_scaler": cluster_scaler,
    "cluster_cols": CLUSTER_COLS,
    "n_clusters": N_CLUSTERS,
    "min_cluster_train": MIN_CLUSTER_TRAIN,
    "log_target": False,
    "global_models": {
        "ridge": global_ridge,
        "xgb": global_xgb,
        "lgb": global_lgb,
        "rf": global_rf,
        "meta": meta_global,
        "blend_weights": w_blend,
    },
    "cluster_bundles": cluster_bundle,
    "feature_columns": FEATURE_COLS,
    "target": TARGET,
    "year_col": YEAR_COL,
}
joblib.dump(payload, OUT_DIR / "hybrid_cluster_bundle.joblib")
print("Saved", OUT_DIR / "hybrid_cluster_bundle.joblib")

with open(OUT_DIR / "hybrid_cluster_feature_columns.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
print("Saved", OUT_DIR / "hybrid_cluster_feature_columns.json")

joblib.dump(
    {"kmeans": kmeans, "cluster_scaler": cluster_scaler, "cluster_cols": CLUSTER_COLS, "n_clusters": N_CLUSTERS},
    OUT_DIR / "hybrid_cluster_kmeans.joblib",
)
print("Saved", OUT_DIR / "hybrid_cluster_kmeans.joblib")

joblib.dump(cluster_bundle, OUT_DIR / "hybrid_cluster_models.joblib")
print("Saved", OUT_DIR / "hybrid_cluster_models.joblib")

snapshot_meta = {
    "version": "v7e",
    "snapshot": _snap,
    "n_clusters": N_CLUSTERS,
    "min_cluster_train": MIN_CLUSTER_TRAIN,
    "cluster_counts_train": pd.Series(cl_train).value_counts().sort_index().to_dict(),
    "cluster_counts_val": pd.Series(cl_val).value_counts().sort_index().to_dict(),
    "cluster_counts_test": pd.Series(cl_test).value_counts().sort_index().to_dict(),
    "n_test_rows_using_global_fallback": int(use_fallback.sum()),
    "test_mape_hybrid": float(evaluate("hybrid", y_test, hybrid_test)["mape"]),
    "test_r2_hybrid": float(evaluate("hybrid", y_test, hybrid_test)["r2"]),
}
with open(OUT_DIR / "hybrid_cluster_meta.json", "w") as f:
    json.dump(snapshot_meta, f, indent=2)
print("Saved", OUT_DIR / "hybrid_cluster_meta.json")

## 7 — Temporal discipline and when clustering helps

- **Temporal discipline:** `StandardScaler` and `KMeans` are fit **only on pre-2024 rows**. Validation (2024) and test (2025+) receive cluster labels via the **fitted** transforms — no refit on future years. The clustering inputs **exclude** `resale_price` and **`transaction_year`**, so segments describe **unit and place** rather than the calendar year.

- **When clustering helps:** If submarkets differ in price–feature relationships, separate stacks can reduce bias. **When it hurts:** Small clusters (below the minimum train size) fall back to the **global** ensemble; noisy cluster boundaries or data sparsity can increase variance. Compare **test MAPE** of the global blend/stack vs the cluster-routed hybrid to see which dominates on held-out 2025+ data.
